In [ ]:
import pandas as pd

df = pd.read_csv('query.csv')
df['time'] = pd.to_datetime(df['time'])

# 1. Filter ke bbox Indonesia
df = df[(df['latitude'] >= -11) & (df['latitude'] <= 6) &
        (df['longitude'] >= 95) & (df['longitude'] <= 141)]

# 2. Kolom tanggal
df['tanggal'] = df['time'].dt.date
df['tahun'] = df['time'].dt.year
df['bulan'] = df['time'].dt.month
df['tahun_bulan'] = df['time'].dt.strftime('%Y-%m')

# 3. Kategori kedalaman
def kategori_kedalaman(d):
    if d < 70:
        return 'Dangkal (<70 km)'
    if d < 300:
        return 'Menengah (70-300 km)'
    return 'Dalam (>=300 km)'
df['kategori_kedalaman'] = df['depth'].apply(kategori_kedalaman)

# 4. Kategori magnitudo
def kategori_mag(m):
    if m < 5:
        return 'M4.5-4.9'
    if m < 6:
        return 'M5.0-5.9'
    if m < 7:
        return 'M6.0-6.9'
    return 'M7.0+'
df['kategori_mag'] = df['mag'].apply(kategori_mag)

# 5. Simpan file bersih
kolom = ['time', 'tanggal', 'tahun', 'bulan', 'tahun_bulan', 'latitude',
         'longitude', 'depth', 'kategori_kedalaman', 'mag', 'kategori_mag',
         'magType', 'place', 'id']
bersih = df[kolom]
bersih.to_csv('gempa_indonesia_bersih.csv', index=False)

# 6. Agregasi bulanan
bulanan = df.groupby('tahun_bulan').agg(
    jumlah_gempa=('mag', 'count'),
    mag_max=('mag', 'max'),
    mag_rata=('mag', 'mean'),
).reset_index()
bulanan.to_csv('gempa_indonesia_bulanan.csv', index=False)

print('baris bersih:', len(bersih))
print('rentang:', bersih['time'].min(), 's/d', bersih['time'].max())
print(bulanan.to_string(index=False))


baris bersih: 2922
rentang: 2024-01-01 05:22:53.884000+00:00 s/d 2026-11-13 04:49:33.549000+00:00
tahun_bulan  jumlah_gempa  mag_max  mag_rata
    2024-01            67      6.7  4.747761
    2024-02            50      5.6  4.764000
    2024-03            72      6.4  4.809722
    2024-04            62      6.4  4.817742
    2024-05            65      6.1  4.790769
    2024-06            70      5.8  4.738571
    2024-07            60      5.5  4.781667
    2024-08            62      5.8  4.759677
    2024-09            53      6.0  4.788679
    2024-10            65      6.1  4.770769
    2024-11            57      5.2  4.721053
    2024-12            73      5.5  4.732877
    2025-01            53      5.7  4.696226
    2025-02            86      6.1  4.797674
    2025-03            82      5.8  4.701220
    2025-04            61      6.2  4.855738
    2025-05            54      5.9  4.801852
    2025-06            54      6.0  4.705556
    2025-07            78      6.7  4.835897
  

In [ ]:
import pandas as pd

df = pd.read_csv('gempa_indonesia_bersih.csv')

# Buang baris yang place-nya bukan Indonesia
buang = ['Philippines', 'Timor Leste', 'Malaysia', 'Palau']
mask_buang = df['place'].str.contains('|'.join(buang), na=False)
print('dibuang:', int(mask_buang.sum()))
df = df[~mask_buang]

df.to_csv('gempa_indonesia_bersih.csv', index=False)

# Regenerasi agregasi bulanan
bulanan = df.groupby('tahun_bulan').agg(
    jumlah_gempa=('mag', 'count'),
    mag_max=('mag', 'max'),
    mag_rata=('mag', 'mean'),
).reset_index()
bulanan.to_csv('gempa_indonesia_bulanan.csv', index=False)

print('baris bersih baru:', len(df))
print('rentang:', df['time'].min(), 's/d', df['time'].max())
print('mag terbesar:', df['mag'].max())
print(bulanan.to_string(index=False))


dibuang: 642
baris bersih baru: 2280
rentang: 2024-01-01 05:22:53.884000+00:00 s/d 2026-11-13 04:49:33.549000+00:00
mag terbesar: 7.8
tahun_bulan  jumlah_gempa  mag_max  mag_rata
    2024-01            52      5.6  4.736538
    2024-02            39      5.6  4.776923
    2024-03            59      6.4  4.816949
    2024-04            50      6.4  4.854000
    2024-05            56      6.1  4.796429
    2024-06            59      5.6  4.732203
    2024-07            50      5.4  4.808000
    2024-08            54      5.8  4.757407
    2024-09            39      6.0  4.794872
    2024-10            52      6.1  4.798077
    2024-11            49      5.2  4.736735
    2024-12            61      5.5  4.732787
    2025-01            41      5.7  4.709756
    2025-02            80      6.1  4.802500
    2025-03            70      5.8  4.704286
    2025-04            46      5.8  4.791304
    2025-05            49      5.9  4.769388
    2025-06            41      5.3  4.670732
    2025-07

In [ ]:
import pandas as pd

# File utama
df = pd.read_csv('gempa_indonesia_bersih.csv')
df['time'] = pd.to_datetime(df['time'], format='mixed', utc=True).dt.strftime('%Y-%m-%d %H:%M:%S')
df.to_csv('gempa_indonesia_tableau.csv', index=False, sep=';', decimal=',')

# File bulanan
bln = pd.read_csv('gempa_indonesia_bulanan.csv')
bln.to_csv('gempa_indonesia_bulanan_tableau.csv', index=False, sep=';', decimal=',')

print('baris utama:', len(df))
print('baris bulanan:', len(bln))
print('contoh time:', df['time'].iloc[0])


baris utama: 2280
baris bulanan: 35
contoh time: 2026-11-13 04:49:33
